In [8]:
import numpy as np
import pandas as pd
import torch

from module.proposed import *

import warnings
warnings.filterwarnings('ignore')

In [9]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

## Model load

In [10]:
model_path = "model/proposed_model.pt" # 용량 문제로 git 업로드 하지 못함, 필요시 다른 방향으로 제공
sample_path = '../data/sample_testloader.npz'

In [11]:
model = ProposedModel(task_num=6, hidden_dim=128, num_heads=8).to(device)
model.load_state_dict(torch.load(model_path))
model.eval();

## Sample data load

In [12]:
sample = np.load(sample_path)
x_all = torch.from_numpy(sample['x']).float()
y_metric = torch.from_numpy(sample['y_metric_order']).float()
metric_order = sample['metric_order'].tolist() # ['VB', 'Ra', 'Fx', 'Fy', 'Fz']
batch_size = 256

print(f"Sample shape - X: {x_all.shape}, y: {y_metric.shape}")
print(f"Target order: {metric_order}")

Sample shape - X: torch.Size([5053, 3, 400]), y: torch.Size([5053, 5])
Target order: ['VB', 'Ra', 'Fx', 'Fy', 'Fz']


## Predict

In [13]:
model_to_metric_idx = [3, 4, 0, 1, 2]
y_pred_all = []

with torch.no_grad():
    for i in range(0, len(x_all), batch_size):
        X = x_all[i:i+batch_size].to(device)
        
        _, mu = model(X)
        mu = torch.cat(mu, dim=1)              # [B, 6]
        
        # metric 순서로 재배열: [VB, Ra, Fx, Fy, Fz]
        pred = mu[:, model_to_metric_idx]      # [B, 5]
        y_pred_all.append(pred.cpu())

y_pred_all = torch.cat(y_pred_all, dim=0).numpy()    # [N, 5]
y_true_all = y_metric.numpy()                         # [N, 5]

## Evaluation

In [14]:
results = {}

for i, var in enumerate(metric_order):
    true = y_true_all[:, i]
    pred = y_pred_all[:, i]
    
    results[var] = {
        'MAE':   np.mean(np.abs(true - pred)),
        'RMSE':  np.sqrt(np.mean((true - pred) ** 2))
    }

print(f"\n{'Task':<8}{'MAE':>12}{'RMSE':>12}")
print("--------------------------------")
for var in metric_order:
    r = results[var]
    print(f"{var:<8}{r['MAE']:>12.4f}{r['RMSE']:>12.4f}")


Task             MAE        RMSE
--------------------------------
VB            1.1038      1.4673
Ra            0.2866      0.6189
Fx            7.6901     11.1787
Fy            4.3465      6.1205
Fz            3.0140      3.7843
